# Türkiye yatırım research notebook'u — OpenBB data layer v9

Bu sürümde asıl değişiklik **veri katmanıdır**.

Amaç:
- OpenBB'yi **birincil veri kaynağı** yapmak
- veri çekmeyi strateji kodundan ayırmak
- normalize / validate / fallback akışını netleştirmek

Akış:
1. Kurulum ve config
2. Yardımcı fonksiyonlar
3. OpenBB-first veri katmanı
4. View / signal üretimi
5. Strateji fonksiyonları
6. Backtest motoru
7. Raporlama

Stratejiler:
- Equal
- HRP
- BL
- HYBRID


In [ ]:
!pip -q install openbb yfinance PyPortfolioOpt plotly pandas numpy scipy scikit-learn openai

In [ ]:
import warnings, os, json
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
import yfinance as yf
import plotly.express as px

from pypfopt.risk_models import CovarianceShrinkage
from pypfopt.hierarchical_portfolio import HRPOpt
from pypfopt.black_litterman import BlackLittermanModel
from pypfopt.efficient_frontier import EfficientFrontier


## 1) Config

Bu sürümde `primary_data_source = 'openbb'` olarak ayarlanmıştır.

Eğer OpenBB başarısız olursa otomatik fallback olarak `yfinance` denenir.


In [ ]:
CONFIG = {
    'stock_tickers': ['THYAO.IS', 'AKBNK.IS', 'EREGL.IS'],
    'fx_ticker': 'USDTRY=X',
    'gold_usd_ticker': 'GC=F',
    'asset_labels': {
        'THYAO.IS': 'THYAO',
        'AKBNK.IS': 'AKBNK',
        'EREGL.IS': 'EREGL',
        'USDTRY=X': 'USDTRY',
        'GOLD_TRY': 'ALTIN_TRY',
    },
    'salary_try': 25000,
    'target_test_months': 6,
    'target_lookback_days': 252,
    'download_period': '10y',
    'min_history_days': 180,
    'max_weight': 0.70,
    'gold_label': 'ALTIN_TRY',
    'gold_min': 0.10,
    'gold_max': 0.55,
    'use_llm_views': False,
    'llm_model': 'gpt-4.1-mini',
    'hybrid_alpha': 0.25,
    'strategies': ['Equal', 'HRP', 'BL', 'HYBRID'],
    'primary_data_source': 'openbb',
    'fallback_data_sources': ['yfinance'],
    'openbb_provider': None,
    'start_date': '2016-01-01',
    'max_missing_ratio': 0.20,
    'max_staleness_days': 14,
}
pd.Series(CONFIG)

## 2) Yardımcı fonksiyonlar

Burada yalnızca format / cleaning / weight normalization yardımcıları vardır.


In [ ]:
def try_fmt(x):
    if pd.isna(x):
        return '-'
    return f'₺{x:,.0f}'.replace(',', 'X').replace('.', ',').replace('X', '.')

def pct_fmt(x):
    if pd.isna(x):
        return '-'
    return f'{x*100:.2f}%'

def style(fig, title):
    fig.update_layout(template='plotly_white', title=title, hovermode='x unified', legend_title_text='')
    return fig

def clean_returns(price_window):
    rets = price_window.pct_change()
    rets = rets.replace([np.inf, -np.inf], np.nan).dropna(how='any')
    if rets.empty:
        return rets
    valid_cols = [c for c in rets.columns if np.isfinite(rets[c]).all() and rets[c].std() > 1e-10]
    if len(valid_cols) == 0:
        return pd.DataFrame(index=rets.index)
    return rets[valid_cols]

def band_gold(weights, gold_label, gold_min, gold_max):
    w = weights.copy().astype(float)
    if gold_label not in w.index:
        return w / w.sum()
    current_gold = float(w[gold_label])
    target_gold = min(max(current_gold, gold_min), gold_max)
    if abs(target_gold - current_gold) < 1e-12:
        return w / w.sum()
    rest = w.drop(gold_label)
    if rest.sum() <= 0:
        w[:] = 0.0
        w[gold_label] = 1.0
        return w
    rest = rest / rest.sum()
    w[gold_label] = target_gold
    w.loc[rest.index] = (1 - target_gold) * rest
    return w / w.sum()

def normalize_weights(w, index):
    w = pd.Series(w).reindex(index).fillna(0.0).astype(float)
    s = w.sum()
    if s <= 0 or not np.isfinite(s):
        return pd.Series(1/len(index), index=index)
    return w / s

## 3) OpenBB-first veri katmanı

Bu bölümün amacı:
- OpenBB'den veri çekmek
- format farklarını normalize etmek
- başarısızsa fallback kullanmak
- kalite kontrolleri yapmak


In [ ]:
def coerce_openbb_output_to_series(output, symbol):
    if output is None:
        return None
    df = None
    if isinstance(output, pd.Series):
        s = output.copy()
        s.name = symbol
        return s
    if isinstance(output, pd.DataFrame):
        df = output.copy()
    elif hasattr(output, 'to_dataframe'):
        df = output.to_dataframe()
    elif hasattr(output, 'to_df'):
        df = output.to_df()
    elif hasattr(output, 'results'):
        try:
            df = pd.DataFrame(output.results)
        except Exception:
            df = None

    if df is None or len(df) == 0:
        return None

    if not isinstance(df.index, pd.DatetimeIndex):
        date_candidates = [c for c in df.columns if str(c).lower() in ['date', 'datetime', 'timestamp']]
        if date_candidates:
            df[date_candidates[0]] = pd.to_datetime(df[date_candidates[0]])
            df = df.set_index(date_candidates[0])

    close_candidates = [c for c in df.columns if str(c).lower() in ['close', 'adj_close', 'adjusted_close']]
    if len(close_candidates) == 0:
        numeric_cols = list(df.select_dtypes(include=[np.number]).columns)
        if len(numeric_cols) == 0:
            return None
        close_col = numeric_cols[0]
    else:
        close_col = close_candidates[0]

    s = pd.Series(df[close_col]).dropna()
    s.name = symbol
    return s if len(s) else None

def validate_series(s, symbol, config):
    issues = []
    if s is None or len(s) == 0:
        return None, ['empty']

    s = pd.Series(s).copy()
    s = s[~s.index.duplicated(keep='last')]
    s = s.sort_index()
    s = s.replace([np.inf, -np.inf], np.nan).dropna()

    if not isinstance(s.index, pd.DatetimeIndex):
        try:
            s.index = pd.to_datetime(s.index)
        except Exception:
            issues.append('bad_index')

    if len(s) < config['min_history_days']:
        issues.append('short_history')

    if len(s) > 1:
        full_range = pd.date_range(s.index.min(), s.index.max(), freq='D')
        missing_ratio = 1 - (len(s.index.unique()) / max(len(full_range), 1))
        if missing_ratio > config['max_missing_ratio']:
            issues.append(f"missing_ratio>{config['max_missing_ratio']:.0%}")

    if len(s) and (pd.Timestamp.today().normalize() - s.index.max().normalize()).days > config['max_staleness_days']:
        issues.append('stale')

    return s, issues

def fetch_series_from_yfinance(symbol, config):
    df = yf.download(symbol, period=config['download_period'], auto_adjust=True, progress=False)
    if df is None or len(df) == 0:
        return None

    if isinstance(df.columns, pd.MultiIndex):
        level0 = list(df.columns.get_level_values(0))
        close = df.xs('Close', axis=1, level=0) if 'Close' in level0 else df.iloc[:, :1]
    else:
        close = df[['Close']] if 'Close' in df.columns else df.iloc[:, :1]

    if isinstance(close, pd.DataFrame):
        if close.shape[1] == 0:
            return None
        close = close.iloc[:, 0]

    s = pd.Series(close).dropna()
    s.name = symbol
    return s if len(s) else None

def fetch_series_from_openbb(symbol, config):
    try:
        from openbb import obb
    except Exception:
        return None

    attempts = []
    if config['start_date'] is not None and config['openbb_provider']:
        attempts.append(lambda: obb.equity.price.historical(symbol, start_date=config['start_date'], provider=config['openbb_provider']))
    if config['start_date'] is not None:
        attempts.append(lambda: obb.equity.price.historical(symbol, start_date=config['start_date']))
    if config['openbb_provider']:
        attempts.append(lambda: obb.equity.price.historical(symbol, provider=config['openbb_provider']))
    attempts.append(lambda: obb.equity.price.historical(symbol))

    for fn in attempts:
        try:
            output = fn()
            s = coerce_openbb_output_to_series(output, symbol)
            if s is not None and len(s):
                return s
        except Exception:
            continue

    return None

def fetch_series(symbol, config):
    provider_order = [config['primary_data_source']] + [p for p in config['fallback_data_sources'] if p != config['primary_data_source']]
    errors = []

    for provider in provider_order:
        s = None
        if provider == 'openbb':
            s = fetch_series_from_openbb(symbol, config)
        elif provider == 'yfinance':
            s = fetch_series_from_yfinance(symbol, config)
        else:
            errors.append(f'unknown_provider:{provider}')
            continue

        s, issues = validate_series(s, symbol, config)
        if s is not None and len(s):
            return s, provider, issues
        errors.extend(issues)

    return None, None, list(dict.fromkeys(errors))

def download_market_series(config):
    rows = []
    downloaded = {}
    symbols = config['stock_tickers'] + [config['fx_ticker'], config['gold_usd_ticker']]

    for symbol in symbols:
        s, provider_used, issues = fetch_series(symbol, config)
        downloaded[symbol] = s
        rows.append({
            'source': symbol,
            'asset': config['asset_labels'].get(symbol, symbol),
            'provider_used': provider_used,
            'n': 0 if s is None else len(s),
            'start': None if s is None else str(s.index.min().date()),
            'end': None if s is None else str(s.index.max().date()),
            'issues': ', '.join(issues) if issues else '',
            'status': 'ok' if s is not None and len(s) else 'empty',
        })

    return downloaded, pd.DataFrame(rows)

def build_gold_try(downloaded, config):
    gold_usd = downloaded.get(config['gold_usd_ticker'])
    usdtry = downloaded.get(config['fx_ticker'])
    if gold_usd is None or usdtry is None:
        return None
    gold_try = (gold_usd * usdtry).dropna()
    gold_try.name = config['asset_labels']['GOLD_TRY']
    gold_try, _issues = validate_series(gold_try, 'GOLD_TRY', config)
    return gold_try if gold_try is not None and len(gold_try) else None

def prepare_price_matrix(downloaded, gold_try, config):
    series = []
    rows = []

    for symbol in config['stock_tickers'] + [config['fx_ticker']]:
        s = downloaded.get(symbol)
        eligible = s is not None and len(s) >= config['min_history_days']
        if eligible:
            s = s.copy()
            s.name = config['asset_labels'][symbol]
            series.append(s)
        rows.append({'asset': config['asset_labels'].get(symbol, symbol), 'n': 0 if s is None else len(s), 'eligible': bool(eligible)})

    gold_eligible = gold_try is not None and len(gold_try) >= config['min_history_days']
    if gold_eligible:
        series.append(gold_try)
    rows.append({'asset': config['asset_labels']['GOLD_TRY'], 'n': 0 if gold_try is None else len(gold_try), 'eligible': bool(gold_eligible)})

    eligible_assets = pd.DataFrame(rows)

    if len(series) < 3:
        raise ValueError('Yeterli tarihçesi olan en az 3 varlık yok.')

    prices = pd.concat(series, axis=1).sort_index().ffill()
    common_start = prices.apply(lambda s: s.first_valid_index()).max()
    prices = prices.loc[common_start:].dropna()

    monthly_points = prices.resample('MS').first().dropna()
    if len(monthly_points) < config['target_test_months'] + 1:
        raise ValueError(f"Yeterli ortak tarihçe yok. Aylık nokta sayısı: {len(monthly_points)}")

    lookback = min(config['target_lookback_days'], max(60, len(prices) - 40))
    months = config['target_test_months']
    return prices, eligible_assets, lookback, months

## 4) View / signal üretimi

BL tarafı burada view üretir. Varsayılan olarak rule-based view kullanılır.


In [ ]:
def views_from_prices(price_window):
    rets = clean_returns(price_window)
    if rets.empty or rets.shape[1] == 0:
        return {c: 0.0 for c in price_window.columns}
    ann = rets.mean() * 252
    vol = rets.std() * np.sqrt(252)
    score = (ann / vol.replace(0, np.nan)).replace([np.inf, -np.inf], np.nan).fillna(0)
    score = score.reindex(price_window.columns).fillna(0.0)
    return score.clip(-0.15, 0.15).to_dict()

def llm_views_from_prices(price_window, model='gpt-4.1-mini'):
    labels = list(price_window.columns)
    rule_views = views_from_prices(price_window)
    api_key = os.getenv('OPENAI_API_KEY')
    if not api_key:
        return rule_views, 'rules_no_key'
    try:
        from openai import OpenAI
        recent = clean_returns(price_window.tail(60))
        if recent.empty:
            return rule_views, 'rules_insufficient_data'
        ann = (recent.mean() * 252).round(4).to_dict()
        vol = (recent.std() * np.sqrt(252)).round(4).to_dict()
        prompt = {
            'task': 'Return annual expected return views for Black-Litterman.',
            'rules': [
                'Return valid JSON only.',
                'Keys must exactly match the asset labels provided.',
                'Values must be floats between -0.15 and 0.15.',
                'Do not add any explanation.'
            ],
            'assets': labels,
            'features': {'annualized_recent_return': ann, 'annualized_volatility': vol},
            'output_example': {k: 0.01 for k in labels},
        }
        client = OpenAI(api_key=api_key)
        resp = client.responses.create(
            model=model,
            input=[
                {'role': 'system', 'content': 'You are a quantitative portfolio assistant. Output strict JSON only.'},
                {'role': 'user', 'content': json.dumps(prompt, ensure_ascii=False)}
            ],
            temperature=0
        )
        text = getattr(resp, 'output_text', None) or str(resp)
        parsed = json.loads(text)
        views = {k: float(parsed.get(k, 0.0)) for k in labels}
        views = {k: float(min(max(v, -0.15), 0.15)) for k, v in views.items()}
        return views, 'llm'
    except Exception as e:
        print('LLM failed, rules fallback kullanıldı:', str(e))
        return rule_views, 'rules_fallback'

def get_views(price_window, config):
    if config['use_llm_views']:
        return llm_views_from_prices(price_window, model=config['llm_model'])
    return views_from_prices(price_window), 'rules'

## 5) Strateji fonksiyonları


In [ ]:
def equal_weight(columns):
    return pd.Series(1 / len(columns), index=columns)

def inverse_vol_weight(price_window):
    rets = clean_returns(price_window)
    if rets.empty or rets.shape[1] == 0:
        return equal_weight(price_window.columns)
    iv = 1 / rets.std().replace(0, np.nan)
    iv = iv.replace([np.inf, -np.inf], np.nan).dropna()
    if iv.empty:
        return equal_weight(price_window.columns)
    w = pd.Series(0.0, index=price_window.columns)
    w.loc[iv.index] = iv / iv.sum()
    return normalize_weights(w, price_window.columns)

def hrp_weights_safe(price_window):
    rets = clean_returns(price_window)
    if rets.shape[0] < 2 or rets.shape[1] < 2:
        return inverse_vol_weight(price_window)
    try:
        hrp = HRPOpt(rets)
        w_sub = pd.Series(hrp.optimize()).reindex(rets.columns).fillna(0.0)
        w = pd.Series(0.0, index=price_window.columns)
        w.loc[w_sub.index] = w_sub
        return normalize_weights(w, price_window.columns)
    except Exception as e:
        print('HRP failed, inverse-vol kullanıldı:', str(e))
        return inverse_vol_weight(price_window)

def bl_weights_safe(price_window, config):
    rets = clean_returns(price_window)
    if rets.shape[0] < 2 or rets.shape[1] < 2:
        w = inverse_vol_weight(price_window)
        diag = pd.DataFrame({'asset': list(price_window.columns), 'view': [0.0]*len(price_window.columns), 'view_source': 'rules_insufficient_data'})
        return w, diag
    try:
        S = CovarianceShrinkage(rets, returns_data=True).ledoit_wolf()
        views, view_source = get_views(price_window, config)
        bl = BlackLittermanModel(S, pi='equal', absolute_views=views, omega='default')
        post = bl.bl_returns()
        ef = EfficientFrontier(post, S, weight_bounds=(0, config['max_weight']))
        ef.max_sharpe(risk_free_rate=0.0)
        w = pd.Series(ef.clean_weights()).reindex(price_window.columns).fillna(0.0)
        w = normalize_weights(w, price_window.columns)
        w = band_gold(w, config['gold_label'], config['gold_min'], config['gold_max'])
        diag = pd.DataFrame({'asset': list(views.keys()), 'view': list(views.values()), 'view_source': view_source})
        return w, diag
    except Exception as e:
        print('BL failed, inverse-vol kullanıldı:', str(e))
        w = inverse_vol_weight(price_window)
        fallback_views = views_from_prices(price_window)
        diag = pd.DataFrame({'asset': list(fallback_views.keys()), 'view': list(fallback_views.values()), 'view_source': 'rules_bl_fallback'})
        return w, diag

def hybrid_weights(price_window, config):
    w_hrp = hrp_weights_safe(price_window)
    w_bl, diag = bl_weights_safe(price_window, config)
    alpha = config['hybrid_alpha']
    w = (1 - alpha) * w_hrp + alpha * w_bl
    w = normalize_weights(w, price_window.columns)
    w = w.clip(lower=0, upper=config['max_weight'])
    w = normalize_weights(w, price_window.columns)
    w = band_gold(w, config['gold_label'], config['gold_min'], config['gold_max'])
    return w, diag

def compute_weights(strategy_name, price_window, config):
    if strategy_name == 'Equal':
        return equal_weight(price_window.columns), None
    if strategy_name == 'HRP':
        return hrp_weights_safe(price_window), None
    if strategy_name == 'BL':
        return bl_weights_safe(price_window, config)
    if strategy_name == 'HYBRID':
        return hybrid_weights(price_window, config)
    raise ValueError(f'Unknown strategy: {strategy_name}')

## 6) Backtest motoru


In [ ]:
def get_rebalance_dates(prices, months):
    monthly = prices.resample('MS').first().dropna()
    month_starts = monthly.index[-months:]
    dates = []
    for dt in month_starts:
        idx = prices.index.searchsorted(dt)
        if idx < len(prices.index):
            dates.append(prices.index[idx])
    return pd.Index(pd.unique(pd.Index(dates)))

def run_strategy(strategy_name, prices, lookback, months, config):
    rebalance_dates = get_rebalance_dates(prices, months)
    cash = 0.0
    shares = pd.Series(0.0, index=prices.columns)
    hist, rebs, diags = [], [], []

    for i, dt in enumerate(rebalance_dates):
        cash += config['salary_try']
        window = prices.loc[:dt].tail(lookback)
        weights, diag = compute_weights(strategy_name, window, config)
        current = prices.loc[dt]
        total = float((shares * current).sum()) + cash
        shares = (weights * total) / current
        cash = 0.0

        nxt = rebalance_dates[i + 1] if i + 1 < len(rebalance_dates) else prices.index[-1]
        path = prices.loc[(prices.index >= dt) & (prices.index <= nxt)]
        values = path.mul(shares, axis=1).sum(axis=1)

        for t, v in values.items():
            hist.append({'date': t, 'strategy': strategy_name, 'portfolio_value': float(v)})

        row = {'date': dt, 'strategy': strategy_name, 'capital_after_contribution': total}
        for c, x in weights.items():
            row[f'weight_{c}'] = float(x)
        rebs.append(row)

        if diag is not None:
            tmp = diag.copy()
            tmp['date'] = dt
            tmp['strategy'] = strategy_name
            diags.append(tmp)

    hist_df = pd.DataFrame(hist).drop_duplicates(['date', 'strategy'])
    weights_df = pd.DataFrame(rebs)
    diag_df = pd.concat(diags, ignore_index=True) if diags else pd.DataFrame()
    return hist_df, weights_df, diag_df

def run_all_strategies(prices, lookback, months, config):
    hist_parts, weight_parts, diag_parts = [], [], []
    for s in config['strategies']:
        h, w, d = run_strategy(s, prices, lookback, months, config)
        hist_parts.append(h)
        weight_parts.append(w)
        if not d.empty:
            diag_parts.append(d)
    hist_df = pd.concat(hist_parts, ignore_index=True)
    weights_df = pd.concat(weight_parts, ignore_index=True)
    diag_df = pd.concat(diag_parts, ignore_index=True) if diag_parts else pd.DataFrame()
    return hist_df, weights_df, diag_df

## 7) Raporlama


In [ ]:
def build_summary(hist_df, months, config):
    rows = []
    contributed = config['salary_try'] * months
    for s in config['strategies']:
        sub = hist_df[hist_df['strategy'] == s].sort_values('date').copy()
        sub['ret'] = sub['portfolio_value'].pct_change().fillna(0.0)
        dd = sub['portfolio_value'] / sub['portfolio_value'].cummax() - 1
        rows.append({
            'Strategy': s,
            'Contributed': try_fmt(contributed),
            'Ending Value': try_fmt(sub['portfolio_value'].iloc[-1]),
            'Net Profit': try_fmt(sub['portfolio_value'].iloc[-1] - contributed),
            'TWR': pct_fmt(sub['portfolio_value'].iloc[-1] / contributed - 1),
            'Ann Vol': pct_fmt(sub['ret'].std() * np.sqrt(252)),
            'Sharpe': f"{(sub['ret'].mean() / (sub['ret'].std() + 1e-9) * np.sqrt(252)):.2f}",
            'MaxDD': pct_fmt(dd.min()),
        })
    return pd.DataFrame(rows)

def build_last_weights(weights_df):
    last = weights_df.sort_values('date').groupby('strategy').tail(1).copy()
    cols = [c for c in last.columns if c.startswith('weight_')]
    out = last[['strategy'] + cols].copy()
    out.columns = ['Strategy'] + [c.replace('weight_', '') for c in cols]
    for c in out.columns[1:]:
        out[c] = out[c].map(pct_fmt)
    return out.reset_index(drop=True)

def plot_market_structure(prices):
    fig = px.line(prices.tail(504) / prices.tail(504).iloc[0] * 100, x=prices.tail(504).index, y=prices.columns)
    style(fig, 'Son ~2 yıl normalize fiyatlar').show()
    fig = px.imshow(prices.pct_change().dropna().corr(), text_auto=True, aspect='auto', color_continuous_scale='RdBu_r', zmin=-1, zmax=1)
    style(fig, 'Korelasyon').show()

def plot_portfolio_values(hist_df):
    fig = px.line(hist_df, x='date', y='portfolio_value', color='strategy')
    fig.update_yaxes(tickprefix='₺', separatethousands=True)
    style(fig, 'Portföy değeri').show()

## 8) Notebook'u çalıştır


In [ ]:
downloaded, coverage = download_market_series(CONFIG)
gold_try = build_gold_try(downloaded, CONFIG)

extra_row = {
    'source': f"{CONFIG['gold_usd_ticker']} * {CONFIG['fx_ticker']}",
    'asset': CONFIG['asset_labels']['GOLD_TRY'],
    'provider_used': 'synthetic',
    'n': len(gold_try) if gold_try is not None else 0,
    'start': str(gold_try.index.min().date()) if gold_try is not None else None,
    'end': str(gold_try.index.max().date()) if gold_try is not None else None,
    'issues': '',
    'status': 'ok' if gold_try is not None else 'empty',
}
coverage = pd.concat([coverage, pd.DataFrame([extra_row])], ignore_index=True)
display(coverage)

prices, eligible_assets, lookback, months = prepare_price_matrix(downloaded, gold_try, CONFIG)
display(eligible_assets)
print('Kullanılan varlıklar:', list(prices.columns))
print('Toplam gün:', len(prices), 'lookback:', lookback, 'test_months:', months)
display(prices.tail())

plot_market_structure(prices)

hist_df, weights_df, diag_df = run_all_strategies(prices, lookback, months, CONFIG)
summary = build_summary(hist_df, months, CONFIG)
display(summary)

plot_portfolio_values(hist_df)

display(build_last_weights(weights_df))
display(weights_df.tail(len(CONFIG['strategies']) * 3))

if len(diag_df):
    diag_show = diag_df.copy()
    diag_show['view'] = diag_show['view'].map(pct_fmt)
    display(diag_show.tail(20))